# Colormap Comparison

Side-by-side comparison of candidate colormaps to replace the default viridis scheme.
The target aesthetic is: **white = no return → blue → green → red = strongest return**.

In [ ]:
import sys
sys.path.insert(0, '../..')
from project_paths import REPO_ROOT

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from raw_sensor_integration.data_loader import load_raw

## Load an echogram

In [ ]:
RAW_DIR = REPO_ROOT / 'Datasets/raw_files/drive-download-20260317T113707Z-1-001'
raw_files = sorted(RAW_DIR.glob('*.raw'))
print(f'Found {len(raw_files)} files')

data = load_raw(raw_files[0], frequency=38000)
power_db = data.sv_db
range_array = data.range_array

print(f'Shape: {power_db.shape}  range: {power_db.min():.0f} – {power_db.max():.0f} dB')

# Clip and set display limits
power_db = np.clip(power_db, -80, -20)
VMIN, VMAX = -80, -20
n_pings = power_db.shape[1]
extent = [0, n_pings, range_array[-1], range_array[0]]

## Define candidate colormaps

All maps are defined so that **low values (weak return) map to white** and **high values (strong return) map to red**.
Each colormap is built as a list of (value, color) anchor points.

In [ ]:
def make_cmap(name, anchors):
    """Build a LinearSegmentedColormap from (position, color) anchors."""
    positions = [a[0] for a in anchors]
    colors    = [a[1] for a in anchors]
    return mcolors.LinearSegmentedColormap.from_list(name, list(zip(positions, colors)))


# 1. Simple white → blue → green → red
cmap_wbgr = make_cmap('wbgr', [
    (0.00, 'white'),
    (0.33, '#0000cc'),   # pure blue
    (0.66, '#00cc00'),   # pure green
    (1.00, '#cc0000'),   # pure red
])

# 2. white → navy → teal → green → yellow → red  (more stops, smoother)
cmap_wbgyr = make_cmap('wbgyr', [
    (0.00, 'white'),
    (0.20, '#00008b'),   # dark blue
    (0.40, '#008080'),   # teal
    (0.60, '#00aa00'),   # green
    (0.80, '#ffdd00'),   # yellow
    (1.00, '#dd0000'),   # red
])

# 3. Classic EK500/EK80 SIMRAD palette (close to the hardware display)
#    dark blue → blue → cyan → green → yellow → orange → red
cmap_ek500 = make_cmap('ek500', [
    (0.00, '#ffffff'),   # white background
    (0.10, '#0000ff'),   # blue
    (0.25, '#00ffff'),   # cyan
    (0.45, '#00ff00'),   # green
    (0.65, '#ffff00'),   # yellow
    (0.80, '#ff8800'),   # orange
    (1.00, '#ff0000'),   # red
])

# 4. Pastel / muted version of wbgr — softer on the eyes
cmap_soft = make_cmap('soft', [
    (0.00, 'white'),
    (0.25, '#a8c8f0'),   # light blue
    (0.50, '#5db85d'),   # muted green
    (0.75, '#e06030'),   # burnt orange
    (1.00, '#9b0000'),   # dark red
])

# 5. High-contrast: white → deep blue → lime → red (sharp transitions)
cmap_hc = make_cmap('highcontrast', [
    (0.00, 'white'),
    (0.15, '#000080'),   # navy
    (0.50, '#00ff80'),   # lime-green
    (1.00, '#ff0000'),   # red
])

# 6. Reference: original viridis (for comparison)
cmap_viridis = plt.get_cmap('viridis')

# Collect all
candidates = [
    (cmap_wbgr,    'wbgr — white→blue→green→red'),
    (cmap_wbgyr,   'wbgyr — white→blue→teal→green→yellow→red'),
    (cmap_ek500,   'ek500 — SIMRAD-style (white background)'),
    (cmap_soft,    'soft — muted palette'),
    (cmap_hc,      'highcontrast — white→navy→lime→red'),
    (cmap_viridis, 'viridis — current (reference)'),
]

print(f'Defined {len(candidates)} colormaps')

## Full echogram — all candidates

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(18, 14))

for ax, (cmap, label) in zip(axes.flatten(), candidates):
    im = ax.imshow(
        power_db, aspect='auto', origin='upper',
        cmap=cmap, vmin=VMIN, vmax=VMAX, extent=extent,
    )
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('Ping')
    ax.set_ylabel('Range (m)')
    plt.colorbar(im, ax=ax, shrink=0.8, label='dB')

plt.suptitle('Colormap candidates — full echogram', fontsize=13)
plt.tight_layout()
plt.show()

## Zoomed view near the seabed

Zoom in on a 400-ping window centred on the bottom to compare how each map renders the seabed boundary.

In [ ]:
# Find approximate bottom depth from the strongest return per ping
bottom_sample = np.argmax(power_db, axis=0)
median_bottom_sample = int(np.median(bottom_sample))

WIN_DEPTH = 80    # samples above and below bottom to show
WIN_PING  = 200   # half-width in pings

d0 = max(0, median_bottom_sample - WIN_DEPTH)
d1 = min(power_db.shape[0], median_bottom_sample + WIN_DEPTH)
p0 = max(0, n_pings // 2 - WIN_PING)
p1 = min(n_pings, n_pings // 2 + WIN_PING)

zoom = power_db[d0:d1, p0:p1]
zoom_extent = [p0, p1, range_array[d1 - 1], range_array[d0]]

print(f'Zoom region: depth samples {d0}–{d1}, pings {p0}–{p1}')

fig, axes = plt.subplots(3, 2, figsize=(18, 14))

for ax, (cmap, label) in zip(axes.flatten(), candidates):
    im = ax.imshow(
        zoom, aspect='auto', origin='upper',
        cmap=cmap, vmin=VMIN, vmax=VMAX, extent=zoom_extent,
    )
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('Ping')
    ax.set_ylabel('Range (m)')
    plt.colorbar(im, ax=ax, shrink=0.8, label='dB')

plt.suptitle('Colormap candidates — zoomed near seabed', fontsize=13)
plt.tight_layout()
plt.show()

## Colorbar strips

Direct comparison of the gradient from weak (left) to strong (right).

In [ ]:
gradient = np.linspace(0, 1, 256).reshape(1, -1)

fig, axes = plt.subplots(len(candidates), 1, figsize=(12, 5))

for ax, (cmap, label) in zip(axes, candidates):
    ax.imshow(gradient, aspect='auto', cmap=cmap)
    ax.set_yticks([])
    ax.set_xticks([0, 64, 128, 192, 255])
    ax.set_xticklabels([f'{VMIN:.0f}', '', f'{(VMIN+VMAX)/2:.0f}', '', f'{VMAX:.0f}'], fontsize=8)
    ax.set_ylabel(label.split('—')[0].strip(), fontsize=9, rotation=0, ha='right', va='center', labelpad=130)

plt.suptitle('Colormap gradient strips  (weak ← → strong)', fontsize=11)
plt.tight_layout()
plt.show()

## Single-ping profile overlay

Shows how each colormap translates the depth profile of one representative ping — useful for judging contrast at the seabed boundary.

In [ ]:
PING = n_pings // 2
profile = power_db[:, PING]
depths = range_array[:len(profile)]

# Normalise to 0–1 for colormap lookup
norm = mcolors.Normalize(vmin=VMIN, vmax=VMAX)

fig, axes = plt.subplots(1, len(candidates), figsize=(20, 8), sharey=True)

for ax, (cmap, label) in zip(axes, candidates):
    # Draw as a scatter so each point gets its colour from the cmap
    sc = ax.scatter(profile, depths, c=profile, cmap=cmap, norm=norm, s=2, linewidths=0)
    ax.set_title(label.split('—')[0].strip(), fontsize=9)
    ax.set_xlabel('dB', fontsize=8)
    ax.set_xlim(VMIN, VMAX)
    ax.invert_yaxis()

axes[0].set_ylabel('Range (m)')
plt.suptitle(f'Single-ping profile (ping {PING}) — colour = cmap value', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

import echopype as ep

# Grab one file from the OOI file server
raw_url =str(REPO_ROOT / 'Datasets/raw_files/drive-download-20260317T120831Z-1-001/D20240201-T002852.raw')
raw_url =str(REPO_ROOT / 'Datasets/raw_files/drive-download-20260317T113707Z-1-001/D20240131-T114241.raw')

# Convert the raw file into an EchoDat object
echodata = ep.open_raw(raw_file=raw_url, sonar_model="EK80")

# Calibrate the raw data to volume backscattering volume (Sv)
ds_Sv = ep.calibrate.compute_Sv(echodata, waveform_mode="BB", encode_mode="complex")

In [ ]:
# Plot all 3 channels for inspection
ds_Sv["Sv"].plot(
    x="ping_time",
    row="channel",
    figsize=(7, 7),
    vmin=-100, 
    vmax=-30,
    cmap="ep.ek500"
)